# kazeval on Kaggle — canonical GPU runs (compute policy 2026-07-04)

One-time Kaggle setup:
1. **Settings → Accelerator**: GPU T4 x2 or P100 (both 16 GB, fp16 — no bf16).
2. **Settings → Internet**: ON (needs a phone-verified account).
3. **Add-ons → Secrets**: add `HF_TOKEN` = a HuggingFace token of an account that has
   accepted the conditions of the gated datasets/models used below
   ([issai/kazqad-retrieval](https://huggingface.co/datasets/issai/kazqad-retrieval),
   and [inceptionai/Llama-3.1-Sherkala-8B-Chat](https://huggingface.co/inceptionai/Llama-3.1-Sherkala-8B-Chat) for the KazMMLU ceiling run).

The repo is cloned from GitHub `main` — **push before launching**, Kaggle sees only what is pushed.
Free quota: 30 GPU-h/week, sessions ≤ 12 h — long runs must checkpoint/finish inside one session.


In [ ]:
!git clone --depth 1 https://github.com/kekeront/qymyzlm.git /kaggle/working/qymyzlm
%pip install -q /kaggle/working/qymyzlm/evallab

import pathlib

from huggingface_hub import login


def hf_token():
    """Kaggle secret HF_TOKEN (interactive runs) or an attached private
    token dataset with hf_token.txt (kernels pushed via the Kaggle API,
    which cannot access UserSecretsClient secrets)."""
    try:
        from kaggle_secrets import UserSecretsClient

        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        hits = sorted(pathlib.Path("/kaggle/input").glob("**/hf_token.txt"))
        if not hits:
            raise RuntimeError(
                "no HF_TOKEN: add a Kaggle secret or attach the private token dataset"
            ) from None
        return hits[0].read_text().strip()


login(hf_token())

## Run 1 — leaderboard-v0 embedding baselines (KazQADRetrieval)

Full-corpus retrieval only (1,929 KazQAD test queries vs the 825,309-passage Kazakh
Wikipedia corpus) for **five off-the-shelf encoders** through the pinned
`KazQADRetrieval` protocol — this is the run that moves the measured-record count
from 1 to 6 and pays down the apparatus freeze. **Keystone first:** re-measure
`intfloat/multilingual-e5-large` (its KazQADRetrieval *measured* record is the
canonical planka), then `BAAI/bge-m3`, `Qwen/Qwen3-Embedding-0.6B`,
`sentence-transformers/LaBSE`, and
`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`.

Each model runs in its own `kazeval.run_retrieval` subprocess on both T4s
(`--device cuda:0,cuda:1`, fp16), writing one committed-style JSON record per model
into `/kaggle/working/results/`. Reranking is NOT part of v0 — retrieval only.

In [ ]:
# Leaderboard v0 (v8) — 5 encoders through the pinned KazQADRetrieval protocol.
#
# Full-corpus retrieval: 1,929 KazQAD test queries vs the 825,309-passage Kazakh
# Wikipedia corpus. Each --model spawns its OWN subprocess of kazeval.run_retrieval,
# which writes one deterministic record {date}__KazQADRetrieval__{slug}.json into
# /kaggle/working/results the instant that model finishes. That per-process design
# is the isolation spine:
#   * a crash / OOM / non-zero exit / hang on model N cannot touch the JSON already
#     written for models 1..N-1 (different files, separate processes);
#   * GPU memory + CUDA context are reclaimed by the OS when each subprocess exits,
#     so one model's OOM does not poison the next model's run.
#
# KEYSTONE FIRST: the mE5-large KazQADRetrieval *measured* record is NOT on disk
# (only a REPORTED KazQAD-HardNeg mE5 row exists). It is the canonical planka we
# must re-measure, so it runs first and its row is safe earliest.
#
# CORRECTNESS GUARD: run_retrieval resolves --model via mteb.get_model first, which
# supplies the correct asymmetric query:/passage: prompts for the e5 family etc. A
# model NOT in mteb's registry falls back to a bare SentenceTransformer running
# WITHOUT prompts -> WRONG retrieval number. run_retrieval only LOGS that (exit 0),
# so a wrong row would silently reach the leaderboard (the kazembed-v5 failure mode).
# We tee+capture output, and if the fallback warning fired we mark the model INVALID
# and DELETE the record it wrote so it can never be collected.

import os
import shlex
import signal
import subprocess
import sys
import threading
import time
from pathlib import Path

RESULTS = Path("/kaggle/working/results")
RESULTS.mkdir(parents=True, exist_ok=True)

# Order is load-bearing: keystone first, then the rest. All five are expected to be
# in mteb's registry (correct prompt regime); the INVALID guard below is the seatbelt.
KEYSTONE = "intfloat/multilingual-e5-large"
MODELS = [
    KEYSTONE,  # canonical planka
    "BAAI/bge-m3",
    "Qwen/Qwen3-Embedding-0.6B",
    "sentence-transformers/LaBSE",
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
]

BATCH_SIZE = 64  # fp16 on one T4 (per-GPU inside the multi-process pool)
BATCH_OVERRIDES = {}  # e.g. {"Qwen/Qwen3-Embedding-0.6B": 32} if a model OOMs
DEVICE = "cuda:0,cuda:1"  # spread the 825k-passage encode over both T4s
DTYPE = "float16"  # T4 has no bf16; run_retrieval hard-fails a bad cast
PER_MODEL_TIMEOUT = 5 * 3600  # s; a full encode is ~1-3h — generous cap so one hung
# model cannot starve the tail of a <=12h session
FALLBACK_MARK = "not in mteb's model registry"  # run_retrieval's WARNING on bad path


def snapshot():
    """path -> mtime for every record on disk (naming-agnostic new-file detection)."""
    return {p: p.stat().st_mtime for p in RESULTS.glob("*.json")}


def run_one(model):
    bs = BATCH_OVERRIDES.get(model, BATCH_SIZE)
    cmd = [
        sys.executable,
        "-m",
        "kazeval.run_retrieval",
        "--model",
        model,
        "--tasks",
        "KazQADRetrieval",  # retrieval only — NOT reranking (v0 is retrieval)
        "--batch-size",
        str(bs),
        "--device",
        DEVICE,
        "--dtype",
        DTYPE,
        "--output-dir",
        str(RESULTS),
    ]
    tag = "KEYSTONE " if model == KEYSTONE else ""
    print(
        f"\n{'=' * 74}\n>>> {tag}{model}  (batch={bs})\n>>> {shlex.join(cmd)}\n{'=' * 74}",
        flush=True,
    )

    before = snapshot()
    t0 = time.time()
    buf = []

    # start_new_session -> the child leads its own process group, so on timeout we can
    # SIGKILL the WHOLE group (the multi-GPU pool spawns workers) instead of leaking
    # orphan encoders into the next model's run.
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        start_new_session=True,
    )

    def pump():
        # tee: stream live to the notebook AND capture for the fallback scan. Line-based;
        # INFO logs and the final score line are newline-terminated (tqdm \r bars may only
        # surface at completion, which is acceptable — the load-bearing lines stream).
        for line in proc.stdout:
            sys.stdout.write(line)
            sys.stdout.flush()
            buf.append(line)

    reader = threading.Thread(target=pump, daemon=True)
    reader.start()
    reader.join(PER_MODEL_TIMEOUT)

    timed_out = reader.is_alive()
    if timed_out:
        try:
            os.killpg(os.getpgid(proc.pid), signal.SIGKILL)
        except ProcessLookupError:
            pass
        proc.wait()
        reader.join(15)
        rc = None
    else:
        rc = proc.wait()

    dur = time.time() - t0
    after = snapshot()
    new_files = [p for p in after if p not in before or after[p] != before.get(p)]
    fell_back = FALLBACK_MARK in "".join(buf)

    # Classify. PASS requires: exit 0, went through mteb's registry (correct prompts),
    # AND actually wrote a record.
    if timed_out:
        status = f"TIMEOUT(>{PER_MODEL_TIMEOUT // 3600}h)"
    elif rc != 0:
        status = f"FAIL(exit {rc})"
    elif fell_back:
        status = "INVALID(no-prompt fallback)"
    elif not new_files:
        status = "FAIL(no record written)"
    else:
        status = "PASS"

    # Quarantine an invalid row: delete the record this model just wrote so the wrong
    # number never reaches the collect/leaderboard step. Only new files, never touch
    # records from earlier models.
    if status.startswith("INVALID"):
        for p in new_files:
            p.unlink(missing_ok=True)
        print(
            f"!!! {model}: fell back to prompt-less SentenceTransformer -> quarantined "
            f"{len(new_files)} invalid record(s); needs a prompt config before it counts",
            flush=True,
        )

    print(
        f"=== {model} -> {status}  ({dur / 60:.1f} min, {len(new_files)} record file(s)) ===",
        flush=True,
    )
    return {
        "model": model,
        "status": status,
        "minutes": dur / 60,
        "files": [p.name for p in new_files if status == "PASS"],
    }


summary = []
for m in MODELS:
    try:
        summary.append(run_one(m))
    except Exception as exc:  # never let one model abort the batch
        print(f"!!! {m}: launcher exception {type(exc).__name__}: {exc}", flush=True)
        summary.append(
            {"model": m, "status": f"LAUNCH-ERR({type(exc).__name__})", "minutes": 0.0, "files": []}
        )

# ---- Per-model pass/fail summary — makes a partial/interrupted session legible ----
print(f"\n{'#' * 74}\n# v8 KazQADRetrieval run summary\n{'#' * 74}")
w = max(len(s["model"]) for s in summary)
n_pass = 0
for s in summary:
    star = " *KEYSTONE*" if s["model"] == KEYSTONE else ""
    ok = s["status"] == "PASS"
    n_pass += ok
    rec = f"  -> {s['files'][0]}" if s["files"] else ""
    print(
        f"  [{'PASS' if ok else 'FAIL'}] {s['model']:<{w}}  {s['status']:<26} "
        f"{s['minutes']:6.1f} min{star}{rec}"
    )
print(f"\n{n_pass}/{len(summary)} model(s) produced a valid KazQADRetrieval row.")

key = next(s for s in summary if s["model"] == KEYSTONE)
if key["status"] != "PASS":
    print(
        f"\n!!! KEYSTONE mE5-large did NOT land ({key['status']}). Re-run it before "
        "trusting this batch — the canonical planka row is still missing on disk."
    )
else:
    print(
        "\nKeystone mE5-large landed. Records are on /kaggle/working/results (Output tab); "
        "run the collect cell, then commit them into evallab/results/."
    )

## Run 2 (optional) — KazMMLU 3-shot, Sherkala-Chat-8B in 4-bit (W5: same-protocol ceiling)

8B in 4-bit ≈ 5.5 GB weights — fits a single T4/P100. Uncomment to run.


In [ ]:
# %pip install -q bitsandbytes
# !python -m kazeval.run_kazmmlu \
#     --model inceptionai/Llama-3.1-Sherkala-8B-Chat \
#     --model-args dtype=float16,load_in_4bit=True \
#     --batch-size auto \
#     --output-dir /kaggle/working/results


## Collect results

Download every JSON under `/kaggle/working/results/` (Output tab), drop the files into
`evallab/results/` in the repo, then locally:

```bash
python -m kazeval.leaderboard   # re-renders the README leaderboard
```


In [ ]:
# Collect — validate every record on /kaggle/working/results, preview the retrieval
# leaderboard, and dump full JSON for download. Validation via kazeval.results.load_record
# catches a half-written file from a SIGKILL'd/timed-out process before it is committed.

from pathlib import Path

from kazeval.results import load_record

RESULTS = Path("/kaggle/working/results")
paths = sorted(RESULTS.glob("*.json"))
print(f"{len(paths)} record file(s) under {RESULTS}\n")

rows, bad = [], []
for p in paths:
    try:
        rows.append(load_record(p))  # schema-validates; rejects malformed/partial JSON
    except Exception as exc:
        bad.append((p.name, str(exc)))

# KazQADRetrieval leaderboard preview (main_score = ndcg_at_10, higher is better).
ret = sorted(
    (r for r in rows if r.task == "KazQADRetrieval"),
    key=lambda r: r.metrics.get("ndcg_at_10", -1.0),
    reverse=True,
)
if ret:
    w = max(len(r.model) for r in ret)
    print(f"{'model':<{w}}  prov      ndcg@10  mrr@10  recall@100")
    for r in ret:
        m = r.metrics
        nan = float("nan")
        print(
            f"{r.model:<{w}}  {r.provenance:<8}  "
            f"{m.get('ndcg_at_10', nan):7.4f}  "
            f"{m.get('mrr_at_10', nan):6.4f}  "
            f"{m.get('recall_at_100', nan):7.4f}"
        )
    print()

# Full JSON so it can be copied out even if the Output download is missed.
for p in paths:
    print(f"----- {p.name} -----")
    print(p.read_text())

if bad:
    print("\n!!! INVALID record files (do NOT commit — re-run these models):")
    for name, err in bad:
        print(f"  {name}: {err}")
else:
    print(
        "\nAll record files validate. Download them (Output tab) into evallab/results/, "
        "then locally: python -m kazeval.leaderboard"
    )